
# Практическое занятие №3 (ПЗ-3)
## Этап 3 процесса Data Science: подготовка данных и воспроизводимый pipeline

**Дисциплина:** Системы обработки больших данных  
**Связь с лекциями:** Лекция 1, Лекция 2  

В этом ноутбуке вы:
- выбираете **свой индивидуальный вариант датасета (1–15)**;
- выполняете **очистку и преобразование данных**;
- собираете все шаги в **воспроизводимый pipeline**;
- сравниваете качество данных **до и после** обработки.

**Важно:** все текстовые ответы пишите прямо в этом ноутбуке в ячейках Markdown.



## Шаг 0. Выбор индивидуального варианта

Введите номер вашего варианта (от 1 до 15).  
Файл должен называться:

`PZ3_PZ4_variant_XX.csv`, где `XX` — номер варианта.


In [ ]:

# >>> ВАЖНО <<<
# Укажите номер вашего варианта (от 1 до 15)

VARIANT = 1  # <<< ИЗМЕНИТЕ НА СВОЙ НОМЕР

# Проверка корректности ввода
assert 1 <= VARIANT <= 15, "Номер варианта должен быть от 1 до 15"

filename = f"PZ3_PZ4_variant_{VARIANT:02d}.csv"
print("Выбран файл:", filename)



## Шаг 1. Загрузка данных и первичный обзор


In [ ]:

import pandas as pd

df_raw = pd.read_csv(filename)
df_raw.head()


In [ ]:

print("Размер датасета:", df_raw.shape)
print("\nТипы данных:")
df_raw.dtypes



###  Ответ студента (обязательно)

Опишите:
1. Что является **объектом наблюдения** (что означает одна строка)?  
2. Какие признаки являются **числовыми**, а какие **категориальными**?



## Шаг 2. План очистки данных



###  План очистки (заполните)

- **Пропуски:**  
- **Дубликаты:**  
- **Выбросы:**  
- **Категориальные признаки:**  
- **Преобразования:**  



## Шаг 3. Анализ пропусков


In [ ]:

missing = df_raw.isna().sum()
missing_percent = missing / len(df_raw) * 100

pd.DataFrame({
    "Пропуски (шт)": missing,
    "Пропуски (%)": missing_percent.round(2)
})



###  Ответ студента

Для **каждого признака с пропусками**:
- укажите выбранную стратегию (mean / median / mode / удаление);
- **обоснуйте** выбор.



## Шаг 4. Поиск и удаление дубликатов


In [ ]:

duplicates = df_raw.duplicated().sum()
print("Количество полных дубликатов:", duplicates)

df_nodup = df_raw.drop_duplicates().copy()
print("Размер после удаления дубликатов:", df_nodup.shape)



###  Ответ студента

Удаляли ли вы дубликаты? Почему?



## Шаг 5. Анализ выбросов


In [ ]:

import matplotlib.pyplot as plt

numeric_cols = df_nodup.select_dtypes(include="number").columns

df_nodup[numeric_cols].boxplot(figsize=(12,6), rot=45)
plt.title("Boxplot числовых признаков")
plt.show()



###  Ответ студента

1. В каких признаках обнаружены выбросы?  
2. Как вы решили с ними поступить и почему?



## Шаг 6. Обработка категориальных признаков


In [ ]:

cat_cols = df_nodup.select_dtypes(include="object").columns
cat_cols



###  Ответ студента

Опишите, какие преобразования вы применили к категориальным признакам  
(приведение регистра, удаление пробелов, объединение значений).



## Шаг 7. Подготовка pipeline


In [ ]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

numeric_features = df_nodup.select_dtypes(include="number").columns.drop("churn")
categorical_features = df_nodup.select_dtypes(include="object").columns

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

preprocessor



###  Ответ студента

Поясните:
- какие признаки обрабатываются как числовые;
- какие — как категориальные;
- что делает каждый шаг pipeline.



## Шаг 8. Применение pipeline и контроль качества


In [ ]:

X = df_nodup.drop(columns="churn")
X_processed = preprocessor.fit_transform(X)

print("Размерность данных до обработки:", X.shape)
print("Размерность данных после обработки:", X_processed.shape)



## Шаг 9. Сравнение качества данных «до / после»


In [ ]:

quality_before = df_raw.isna().sum().sum()
quality_after = pd.DataFrame(X_processed).isna().sum().sum()

print("Общее число пропусков ДО обработки:", quality_before)
print("Общее число пропусков ПОСЛЕ обработки:", quality_after)



### Итоговый вывод

1. Улучшилось ли качество данных?  
2. Какие проблемы остались нерешёнными?  
3. Почему pipeline удобнее ручной обработки?



## Контрольные вопросы

Ответьте письменно:

1. Почему этап подготовки данных считается самым трудоёмким в проектах Data Science?  
2. В каких случаях автоматизация подготовки данных принципиально важна?



## Что будет дальше

Результаты этого ноутбука **обязательно сохраняйте**.  
В Практическом занятии №4 вы будете использовать **эти же данные и pipeline** для:
- EDA (исследовательского анализа данных),
- построения baseline-модели,
- оценки качества и оформления мини-отчёта.
